# Pipeline RAG sur les événements OpenAgenda

Ce notebook construit, de bout en bout, un assistant de questions-réponses sur des événements publics :

1. **Récupération** de 200 événements à venir (API publique, sans clé)
2. **Normalisation** : schéma Pydantic, nettoyage du HTML, dates structurées, tarif fiable
3. **Documents LangChain** : un texte lisible + des métadonnées par événement
4. **Exécution et contrôle** de la conversion
5. **Index FAISS** : embeddings locaux, sauvegarde sur disque
6. **Recherche** sémantique avec filtres
7. **Génération** de réponses en français avec OpenAI

> Exécuter les cellules **dans l'ordre**, avec le noyau `.venv` du projet.

## 1. Récupération des données

On télécharge les événements depuis le jeu de données public OpenAgenda hébergé sur OpenDataSoft, puis on les enregistre tels quels (données brutes) dans `data/events.json`.

In [32]:
# --- Imports : uniquement la bibliothèque standard (pas de dépendance pour cette étape) ---
import json
import urllib.parse   # construire la query string (encodage des caractères spéciaux)
import urllib.request  # faire la requête HTTP
from pathlib import Path

# --- Paramètres ---
# Jeu de données public OpenAgenda hébergé sur OpenDataSoft (pas de clé requise)
URL = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/evenements-publics-openagenda/records"
TOTAL = 200       # nombre d'événements voulus
PAGE_SIZE = 100   # limite max de l'API par appel : on pagine donc par blocs de 100
OUTPUT = Path("data/events.json")  # fichier où l'on sauvegarde les données brutes


def fetch_events(total: int = TOTAL) -> list[dict]:
    """Récupère `total` événements en paginant (limit/offset) jusqu'à en avoir assez."""
    events: list[dict] = []
    while len(events) < total:
        params = urllib.parse.urlencode(
            {
                # on ne demande que ce qui manque (ex. 100 puis 100 pour TOTAL=200)
                "limit": min(PAGE_SIZE, total - len(events)),
                # offset = nombre d'événements déjà reçus : on reprend à la suite
                "offset": len(events),
                # événements à venir, en écartant les dates aberrantes de la source
                # (certains événements sont saisis en l'an 2503 ou 2035)
                "where": "firstdate_end >= now() AND firstdate_begin < date'2028-01-01'",
                # du plus proche au plus lointain : ordre stable entre deux pages
                "order_by": "firstdate_begin asc",
            }
        )
        with urllib.request.urlopen(f"{URL}?{params}", timeout=30) as resp:
            results = json.load(resp)["results"]
        if not results:  # plus rien à lire côté API : on s'arrête, même si total n'est pas atteint
            break
        events.extend(results)
    return events


# --- Exécution : téléchargement puis sauvegarde des données brutes ---
events_raw = fetch_events()
OUTPUT.parent.mkdir(exist_ok=True)  # crée le dossier data/ s'il n'existe pas
OUTPUT.write_text(json.dumps(events_raw, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"{len(events_raw)} événements enregistrés dans {OUTPUT}")

200 événements enregistrés dans data/events.json


## 2. Schéma et nettoyage

On définit la **forme attendue** d'un événement (modèles Pydantic), puis les fonctions qui nettoient le texte et interprètent le tarif.

Règle importante : un prix absent ne veut **pas** dire « gratuit ». Le tarif a donc trois états : `free`, `paid` et `unknown`.

In [33]:
# --- Imports ---
import json
import re                          # expressions régulières (tarif, espaces)
from datetime import datetime
from enum import StrEnum           # énumération dont les valeurs sont des chaînes
from pathlib import Path

from bs4 import BeautifulSoup      # retire les balises HTML des descriptions
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter  # découpe les textes longs
from pydantic import BaseModel, field_validator  # validation des données

# --- Paramètres de découpage ---
MAX_DESCRIPTION_CHARS = 1000  # au-delà, la description est découpée en chunks
CHUNK_OVERLAP = 100           # chevauchement entre deux chunks (évite de couper une idée en deux)


# ============================ Modèles de données ============================

class PriceStatus(StrEnum):
    """Les trois états possibles du tarif."""
    FREE = "free"
    PAID = "paid"
    UNKNOWN = "unknown"  # information absente ou ambiguë : ne jamais lire comme « gratuit »


class Price(BaseModel):
    status: PriceStatus = PriceStatus.UNKNOWN  # par défaut : on ne sait pas
    details: str | None = None  # texte d'origine nettoyé, conservé pour l'affichage


class Event(BaseModel):
    """Événement normalisé. Un champ `None` signifie « information inconnue »."""
    id: str
    title: str
    description: str | None = None
    start: datetime | None = None   # vrais objets datetime : utilisables pour filtrer
    end: datetime | None = None
    venue: str | None = None        # nom du lieu
    address: str | None = None
    city: str | None = None
    postal_code: str | None = None
    department: str | None = None
    region: str | None = None
    latitude: float | None = None
    longitude: float | None = None
    category: str | None = None     # jamais renseignée dans cette source
    keywords: list[str] = []
    price: Price = Price()
    url: str | None = None          # lien source, utile pour citer l'événement

    @field_validator("title")
    @classmethod
    def title_not_empty(cls, v: str) -> str:
        """Rejette un événement sans titre : il ne serait pas exploitable pour une citation."""
        if not v:
            raise ValueError("titre vide")
        return v


# ================================ Nettoyage =================================

def clean_html(raw: str | None) -> str | None:
    """Retire le HTML en gardant les sauts de paragraphe/liste, normalise les espaces."""
    if not raw:
        return None
    soup = BeautifulSoup(raw, "html.parser")
    for br in soup.find_all("br"):
        br.replace_with("\n")          # <br> -> saut de ligne
    for li in soup.find_all("li"):
        li.insert_before("\n- ")       # <li> -> puce markdown
    # séparateur "\n" entre blocs seulement s'il y a des <p>/<div>, sinon texte continu
    text = soup.get_text("\n" if soup.find(["p", "div"]) else "")
    text = text.replace("\xa0", " ")   # espace insécable -> espace normal
    # espaces multiples -> un seul, et on retire les espaces en début/fin de ligne
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.splitlines()]
    # au maximum une ligne vide entre deux paragraphes
    text = re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()
    return text or None


def clean_text(raw: str | None) -> str | None:
    """Pour les champs courts (titre, ville...) : nettoie le HTML puis tout sur une seule ligne."""
    text = clean_html(raw)
    return re.sub(r"\s+", " ", text) if text else None


# ================================== Tarif ===================================

# Gratuit SEULEMENT si le texte entier se réduit à "gratuit" / "entrée libre"
_FREE = re.compile(r"^(l')?(entrée\s+)?(libre|gratuite?)(\s+et\s+(libre|gratuite?))?\.?$", re.I)
# Payant si un montant accompagne une devise : "390€", "CHF 500", "12 euros"...
_PAID = re.compile(r"\d\s*(€|euros?|chf|\$|£)|(€|chf)\s*\d", re.I)


def parse_price(conditions: str | None) -> Price:
    """Gratuit seulement si le texte l'affirme sans ambiguïté ; payant si un montant apparaît.

    « Inscription gratuite mais obligatoire » ou un texte sans montant restent UNKNOWN.
    """
    text = clean_text(conditions)
    if not text:
        return Price()  # aucune info : UNKNOWN (surtout pas FREE)
    if _FREE.match(text.strip()):
        return Price(status=PriceStatus.FREE, details=text)
    if _PAID.search(text):
        return Price(status=PriceStatus.PAID, details=text)
    # texte présent mais ni clairement gratuit ni chiffré : on le garde, statut inconnu
    return Price(details=text)

In [34]:
def _parse_dt(value: str | None) -> datetime | None:
    """Convertit une date ISO 8601 (texte) en objet datetime, ou None si absente."""
    return datetime.fromisoformat(value) if value else None


def normalize_event(raw: dict) -> Event:
    """Transforme un événement brut de l'API en `Event` validé."""
    # --- Description : on préfère la version longue, sinon la courte ---
    long_desc = clean_html(raw.get("longdescription_fr"))
    short_desc = clean_html(raw.get("description_fr"))
    description = long_desc or short_desc
    # si la courte apporte une info absente de la longue, on les combine
    if long_desc and short_desc and short_desc not in long_desc:
        description = f"{short_desc}\n\n{long_desc}"

    # --- Coordonnées GPS : dictionnaire {"lat": ..., "lon": ...} (peut être absent) ---
    coords = raw.get("location_coordinates") or {}
    # --- Catégorie : peut être une liste selon les sources, on la met à plat ---
    category = raw.get("category")
    if isinstance(category, list):
        category = ", ".join(map(str, category))

    return Event(
        id=str(raw["uid"]),
        title=clean_text(raw.get("title_fr")) or "",  # "" déclenche l'erreur « titre vide »
        description=description,
        start=_parse_dt(raw.get("firstdate_begin")),   # premier début
        end=_parse_dt(raw.get("lastdate_end")),        # dernière fin (événements récurrents)
        venue=clean_text(raw.get("location_name")),
        address=clean_text(raw.get("location_address")),
        city=clean_text(raw.get("location_city")),
        postal_code=clean_text(raw.get("location_postalcode")),
        department=clean_text(raw.get("location_department")),
        region=clean_text(raw.get("location_region")),
        latitude=coords.get("lat"),
        longitude=coords.get("lon"),
        category=clean_text(category),
        # mots-clés nettoyés un par un, les vides sont écartés
        keywords=[k for k in (clean_text(k) for k in raw.get("keywords_fr") or []) if k],
        price=parse_price(raw.get("conditions_fr")),   # champ « conditions » = tarif
        url=raw.get("canonicalurl"),
    )
    # NB : les coordonnées des contributeurs (e-mail, téléphone) ne sont volontairement pas reprises.

## 3. Documents LangChain

Chaque événement devient un `Document` : un **texte lisible** (ce qui est vectorisé et envoyé au LLM) et des **métadonnées** (pour filtrer et citer les sources).

Un seul document par événement, sauf les descriptions trop longues, découpées en plusieurs morceaux qui gardent tous l'identifiant et les métadonnées.

In [35]:
def _fmt_date(dt: datetime | None) -> str | None:
    """Date lisible en français : 05/10/2026 09:00."""
    return dt.strftime("%d/%m/%Y %H:%M") if dt else None


def _header(event: Event) -> str:
    """Construit l'en-tête lisible, répété dans chaque document de l'événement."""
    # --- Quand : « début → fin » seulement si la fin diffère du début ---
    when = _fmt_date(event.start)
    if when and event.end and event.end != event.start:
        when += f" → {_fmt_date(event.end)}"
    # --- Lieu : nom, adresse, ville, sans répéter une partie déjà contenue dans une autre ---
    place_parts: list[str] = []
    for part in (event.venue, event.address, event.city):
        if part and not any(part in seen for seen in place_parts):
            place_parts.append(part)
    place = ", ".join(place_parts)
    # --- Tarif : « Non précisé » (et non « gratuit ») quand l'information manque ---
    if event.price.status is PriceStatus.FREE:
        price = "Gratuit"
    elif event.price.details:
        price = event.price.details
    else:
        price = "Non précisé"
    lines = [
        f"Titre : {event.title}",
        f"Quand : {when}" if when else None,
        f"Lieu : {place}" if place else None,
        f"Catégorie : {event.category}" if event.category else None,
        f"Mots-clés : {', '.join(event.keywords)}" if event.keywords else None,
        f"Tarif : {price}",
        f"Source : {event.url}" if event.url else None,
    ]
    return "\n".join(line for line in lines if line)  # les lignes None sont ignorées


def _metadata(event: Event) -> dict:
    """Métadonnées plates (scalaires uniquement), compatibles avec les filtres des vector stores."""
    meta = {
        "event_id": event.id,      # identifiant : permet de regrouper les chunks d'un événement
        "title": event.title,
        "start": event.start.isoformat() if event.start else None,
        "end": event.end.isoformat() if event.end else None,
        "start_ts": int(event.start.timestamp()) if event.start else None,  # nombre : filtres « après le... »
        "end_ts": int(event.end.timestamp()) if event.end else None,
        "venue": event.venue,
        "city": event.city,
        "postal_code": event.postal_code,
        "department": event.department,
        "region": event.region,
        "latitude": event.latitude,
        "longitude": event.longitude,
        "category": event.category,
        "keywords": ", ".join(event.keywords) or None,  # liste -> texte (pas de listes en métadonnées)
        "price_status": event.price.status.value,
        "price_details": event.price.details,
        "source": event.url,       # utilisé pour citer l'événement dans les réponses
    }
    # on supprime les valeurs inconnues : une clé absente = information inconnue
    return {k: v for k, v in meta.items() if v is not None}


# Découpeur : coupe de préférence aux paragraphes, puis aux phrases, puis aux mots
_splitter = RecursiveCharacterTextSplitter(
    chunk_size=MAX_DESCRIPTION_CHARS, chunk_overlap=CHUNK_OVERLAP
)


def event_to_documents(event: Event) -> list[Document]:
    """Un événement -> 1 document (cas courant) ou plusieurs si la description est trop longue."""
    header = _header(event)
    description = event.description
    parts = _splitter.split_text(description) if description else []
    if len(parts) <= 1:  # cas courant : un seul document par événement
        text = f"{header}\n\nDescription : {description}" if description else header
        return [Document(page_content=text, metadata=_metadata(event) | {"chunk_index": 0, "chunk_count": 1})]
    # description longue : un document par morceau, avec en-tête et métadonnées répétés
    return [
        Document(
            page_content=f"{header}\n\nDescription (partie {i + 1}/{len(parts)}) : {part}",
            metadata=_metadata(event) | {"chunk_index": i, "chunk_count": len(parts)},
        )
        for i, part in enumerate(parts)
    ]

## 4. Exécution et contrôle

On applique la normalisation à tous les événements bruts. Un événement invalide est **rejeté et signalé**, sans bloquer les autres.

In [36]:
documents, errors = [], []
for raw in events_raw:
    try:
        # brut -> Event validé -> 1 ou plusieurs Documents LangChain
        documents.extend(event_to_documents(normalize_event(raw)))
    except Exception as exc:
        # événement invalide (ex. titre vide) : on le note et on continue avec les suivants
        errors.append(f"{raw.get('uid')}: {exc}")

# nombre d'événements distincts (un événement peut produire plusieurs documents)
event_ids = {d.metadata["event_id"] for d in documents}
print(f"{len(event_ids)} événements -> {len(documents)} documents, {len(errors)} rejetés")
print(errors)  # détail des rejets

200 événements -> 210 documents, 0 rejetés
[]


In [37]:
# Contrôle visuel : texte lisible du premier document, puis ses métadonnées
# (la dernière expression d'une cellule est affichée automatiquement par Jupyter)
print(documents[0].page_content)
documents[0].metadata

Titre : NDK Festival 2026
Quand : 07/10/2026 13:00 → 11/10/2026 12:00
Lieu : Caen, 9 Cours Caffarelli, 14000 Caen, France
Mots-clés : electro
Tarif : De 6.3 € à 62.68 €
Source : https://openagenda.com/shotgun/events/ndk-festival-2026

Description : NDK FESTIVAL 🔥 6ème ÉDITION MERCREDI 7, JEUDI 8, VENDREDI 9, SAMEDI 10, DIMANCHE 11 CAEN 🌱 Mercredi 7 octobre : Le Son Vert 🐦‍🔥 Mercredi 7 octobre : Holeteeth 🎪 Jeudi 8 octobre : Antilogic /…

NDK FESTIVAL 🔥 6ème ÉDITION

MERCREDI 7, JEUDI 8, VENDREDI 9, SAMEDI 10, DIMANCHE 11

CAEN

🌱 Mercredi 7 octobre : Le Son Vert

🐦‍🔥 Mercredi 7 octobre : Holeteeth

🎪 Jeudi 8 octobre : Antilogic / Borokov Borokov / DJ Startup / DJ Travella / NE2L

🪩 Vendredi 9 octobre : b3bz / CAIVA / cholitx / Digé M0m0 / Ellen Allien / LAZE b2b LEGIT GIRL DJ / nextdimensional / NHM & LABN / Rosa Pistola / s8jfou

🪩 Samedi 10 octobre : Akira / Andy Garvey / Gabber Eleganza b2b Vilain / GTI / MA RI / Masto GTZ b2b Pample'mouss / Nurse Erica / Papa Nugs / Paul Seul & Kra

{'event_id': '86272739',
 'title': 'NDK Festival 2026',
 'start': '2026-10-07T13:00:00+00:00',
 'end': '2026-10-11T12:00:00+00:00',
 'start_ts': 1791378000,
 'end_ts': 1791720000,
 'venue': 'Caen',
 'city': 'Caen',
 'postal_code': '14000',
 'department': 'Calvados',
 'region': 'Normandie',
 'latitude': 49.181132,
 'longitude': -0.347264,
 'keywords': 'electro',
 'price_status': 'paid',
 'price_details': 'De 6.3 € à 62.68 €',
 'source': 'https://openagenda.com/shotgun/events/ndk-festival-2026',
 'chunk_index': 0,
 'chunk_count': 1}

## 5. Index FAISS

Les documents sont transformés en vecteurs (**embeddings**) puis stockés dans un index **FAISS** qui permet de retrouver les plus proches d'une question.

Embeddings locaux `multilingual-e5-small` : gratuits, sans clé, bons en français. Le modèle E5 demande les préfixes `passage:` (documents) et `query:` (questions).

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

INDEX_DIR = "data/faiss_index"  # dossier de sauvegarde de l'index


class E5Embeddings(HuggingFaceEmbeddings):
    """E5 attend un préfixe différent pour les documents et les requêtes.

    Sans ces préfixes, la qualité de la recherche baisse nettement.
    """

    def embed_documents(self, texts):
        # textes indexés -> préfixe « passage: »
        return super().embed_documents([f"passage: {t}" for t in texts])

    def embed_query(self, text):
        # question de l'utilisateur -> préfixe « query: »
        return super().embed_query(f"query: {text}")


# Modèle multilingue léger, téléchargé au premier lancement puis mis en cache
embeddings = E5Embeddings(
    model_name="intfloat/multilingual-e5-small",
    encode_kwargs={"normalize_embeddings": True},  # vecteurs de norme 1 : comparaison plus stable
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20098.88it/s]


In [ ]:
# Calcule l'embedding de chaque document et construit l'index FAISS (quelques secondes)
vectorstore = FAISS.from_documents(documents, embeddings)
# Sauvegarde sur disque : évite de tout recalculer à chaque session
vectorstore.save_local(INDEX_DIR)
print(f"{vectorstore.index.ntotal} vecteurs indexés, sauvegardés dans {INDEX_DIR}")

210 vecteurs indexés, sauvegardés dans data/faiss_index


## 6. Recherche

Recherche sémantique : on trouve les événements dont le **sens** est proche de la question, même sans mot en commun. Le score est une distance : plus il est petit, plus le résultat est proche.

In [ ]:
def search(query, k=3, **filters):
    """Recherche sémantique ; `filters` filtre sur les métadonnées (ex. city='Paris')."""
    # filter=None si aucun filtre n'est donné
    results = vectorstore.similarity_search_with_score(query, k=k, filter=filters or None)
    for doc, score in results:
        m = doc.metadata
        # score | titre | ville | date de début (10 premiers caractères = AAAA-MM-JJ)
        print(f"{score:.3f} | {m['title']} | {m.get('city', '?')} | {m.get('start', '?')[:10]}")
        print(f"        {m.get('source')}")
    return results


# « _ = » évite que Jupyter réaffiche la liste retournée
_ = search("randonnée nature en Bretagne")

0.341 | Si vous cherchez du bouleau, y'en a dans les bois ! | Le Cheylas | 2026-10-08
        https://openagenda.com/francetravail/events/si-vous-cherchez-du-bouleau-yen-a-dans-les-bois
0.355 | Les Journées des Métiers de l'Agriculture et du Vivant Voyage en Agriculture - Visite d'entreprise | Ploërmel | 2026-10-08
        https://openagenda.com/francetravail/events/voyage-en-agriculture-9888502
0.365 | RECRUTEMENT RANDSTAD BTP : PROFILS ELECTRICIEN (débutant ou non) | Boulogne-sur-Mer | 2026-10-08
        https://openagenda.com/francetravail/events/recrutement-randstad-btp-profils-electricien-debutant-ou-non


In [ ]:
# Rechargement de l'index depuis le disque (utile au démarrage d'une nouvelle session,
# sans recalculer les embeddings). allow_dangerous_deserialization est acceptable ici
# car on charge un fichier que l'on a produit soi-même.
vectorstore = FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)
_ = search("artificial intelligence", k=3)  # la recherche fonctionne aussi dans une autre langue

0.356 | Salon emploi dédié aux Femmes : construire le parcours professionnel qui vous ressemble ! | Nantes | 2026-10-08
        https://openagenda.com/francetravail/events/salon-emploi-dedie-aux-femmes-construire-le-parcours-professionnel-qui-vous-ressemble-2050310
0.365 | L’édition Aurora marque un tournant ! | Tourcoing | 2026-10-08
        https://openagenda.com/region-hauts-de-france/events/ledition-aurora-marque-un-tournant
0.369 | Mesdames, participez au Tech Boost Her et découvrez les métiers d'avenir qui se féminisent | Tours | 2026-10-08
        https://openagenda.com/francetravail/events/mesdames-participez-au-tech-boost-her-et-decouvrez-les-metiers-davenir-qui-se-feminisent


## 7. Génération de réponses avec OpenAI

Le LLM reçoit les événements retrouvés comme **contexte** et rédige une réponse naturelle en français. Il est appelé via l'[API OpenAI](https://platform.openai.com/docs). Renseigner `OPENAI_API_KEY` (et éventuellement `OPENAI_MODEL`) dans `.env`.

In [ ]:
import os
from datetime import date

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Charge les variables du fichier .env (clé d'API) ; override=True remplace celles déjà définies
load_dotenv(override=True)

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),  # modèle choisi dans .env, sinon gpt-4o-mini
    api_key=os.environ["OPENAI_API_KEY"],             # KeyError si la clé n'est pas définie
    temperature=0.2,  # faible : réponses fidèles au contexte, peu d'invention
)

# Consignes données au LLM : réponse fondée UNIQUEMENT sur le contexte (anti-hallucination)
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Tu es un assistant qui renseigne sur des événements culturels et publics.\n"
     "Réponds en français, de façon naturelle et concise, en t'appuyant UNIQUEMENT sur les "
     "événements du contexte. Si le contexte ne permet pas de répondre, dis-le clairement.\n"
     "Pour chaque événement cité, indique le titre, la date, le lieu et le lien source.\n"
     "Si le tarif est « Non précisé », ne dis jamais que l'événement est gratuit.\n"
     "Date du jour : {today}."),  # la date permet de comprendre « demain », « ce week-end »
    ("human", "Contexte :\n{context}\n\nQuestion : {question}"),
])

In [ ]:
def retrieve_events(question, k=5, fetch_k=20, **filters):
    """Recherche FAISS puis regroupe les morceaux par événement.

    Renvoie une liste de (document, score) : un seul document par event_id.
    """
    # FAISS applique le filtre APRÈS la recherche vectorielle : avec un filtre, on examine
    # donc tout l'index, sinon on rate les événements qui correspondent mais sont moins proches
    if filters:
        fetch_k = vectorstore.index.ntotal
    # on lit plus de résultats que nécessaire car plusieurs morceaux d'un même événement
    # peuvent apparaître : on les dédoublonne ensuite
    results = vectorstore.similarity_search_with_score(
        question, k=fetch_k, fetch_k=fetch_k, filter=filters or None
    )
    seen, hits = set(), []
    for doc, score in results:
        if doc.metadata["event_id"] not in seen:  # on garde le meilleur morceau de chaque événement
            seen.add(doc.metadata["event_id"])
            hits.append((doc, float(score)))
        if len(hits) == k:
            break
    return hits


def generate_answer(question, docs):
    """Demande au LLM de répondre à la question à partir des documents fournis."""
    if not docs:
        return "Je n'ai trouvé aucun événement correspondant."
    # le contexte = les textes des événements, séparés par un trait
    context = "\n\n---\n\n".join(d.page_content for d in docs)
    # remplit le prompt : date du jour, contexte et question
    messages = prompt.invoke({"today": date.today().strftime("%d/%m/%Y"), "context": context, "question": question})
    return llm.invoke(messages).content  # appel à l'API OpenAI (payant, quota requis)


def answer(question, k=3, **filters):
    """Question -> événements les plus proches -> réponse du LLM. Renvoie (texte, documents sources)."""
    docs = [doc for doc, _ in retrieve_events(question, k=k, **filters)]
    return generate_answer(question, docs), docs


# Exemple complet : la réponse du LLM, puis la liste des liens sources
reply, sources = answer("Y a-t-il des événements en lien avec l'intelligence artificielle?")
print(reply)
print("\nSources :", *[d.metadata["source"] for d in sources], sep="\n- ")

Oui, il y a un événement en lien avec l'intelligence artificielle :

**Titre : L’édition Aurora marque un tournant !**  
**Quand : 08/10/2026 06:30 → 08/10/2026 17:00**  
**Lieu : EuraCreative, 99 A Boulevard Constantin Descat 59200 Tourcoing, France**  
**Tarif : Non précisé**  
**Source :** [Lien](https://openagenda.com/region-hauts-de-france/events/ledition-aurora-marque-un-tournant)

Cet événement se concentre sur l'innovation, le design et l'intelligence artificielle.

Sources :
- https://openagenda.com/region-hauts-de-france/events/ledition-aurora-marque-un-tournant
- https://openagenda.com/francetravail/events/mesdames-participez-au-tech-boost-her-et-decouvrez-les-metiers-davenir-qui-se-feminisent
- https://openagenda.com/france-belgique-calendrier/events/atelier-conference-polarisation-bulles-cognitives-et-confrontation-positive


## 8. API FastAPI et Swagger (lancée depuis le terminal)

L'API est définie dans `api.py` (routes) et `rag.py` (moteur RAG) et se lance **dans le terminal**, pas dans le notebook :

```bash
uv run uvicorn api:app --reload
```

- Swagger (interface interactive) : http://127.0.0.1:8000/docs
- ReDoc : http://127.0.0.1:8000/redoc

| Méthode | Route | Rôle |
|---------|-------|------|
| `GET` | `/health` | État du service (taille de l'index, clé OpenAI) |
| `GET` | `/search` | Recherche sémantique sans LLM, avec filtres |
| `POST` | `/ask` | Réponse en français générée par OpenAI + sources |
| `POST` | `/reindex` | Reconstruit l'index depuis `data/events.json` |

L'API charge l'index sauvegardé dans `data/faiss_index` : exécuter d'abord les sections 1 à 5 pour le créer. Les cellules ci-dessous sont un **client** qui interroge le serveur déjà démarré dans le terminal.

In [ ]:
import json
import urllib.error
import urllib.parse
import urllib.request

BASE = "http://127.0.0.1:8000"  # adresse du serveur uvicorn lancé dans le terminal


def call(method, path, params=None, body=None):
    """Petit client HTTP : renvoie (code HTTP, JSON)."""
    url = BASE + path + ("?" + urllib.parse.urlencode(params) if params else "")
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(url, data=data, method=method, headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            return resp.status, json.load(resp)
    except urllib.error.HTTPError as exc:  # 4xx / 5xx : on renvoie aussi le détail de l'erreur
        return exc.code, json.load(exc)
    except urllib.error.URLError:
        raise SystemExit("Serveur injoignable : lancer `uv run uvicorn api:app --reload` dans le terminal")


print(call("GET", "/health"))

(200, {'status': 'ok', 'indexed_chunks': 210, 'llm_configured': True})


In [ ]:
# Recherche sans LLM (gratuite), avec un filtre sur une ville présente dans l'index
city = documents[0].metadata["city"]
status, hits = call("GET", "/search", {"q": "emploi", "k": 3, "city": city})
print("filtre ville =", city)
print(status, [(h["title"][:40], h["city"], round(h["score"], 3)) for h in hits])

filtre ville = Caen
200 [('NDK Festival 2026', 'Caen', 0.394)]


In [ ]:
# Question complète (RAG) : appelle OpenAI, nécessite une clé valide et du crédit
status, result = call("POST", "/ask", body={"question": "Y a-t-il des événements sur l'intelligence artificielle ?", "k": 3})
print(status)
print(result["answer"] if status == 200 else result)

200
Oui, il y a un événement sur l'intelligence artificielle :

**Titre : L’édition Aurora marque un tournant !**  
**Quand : 08/10/2026 06:30 → 08/10/2026 17:00**  
**Lieu : EuraCreative, 99 A Boulevard Constantin Descat 59200 Tourcoing, France**  
**Tarif : Non précisé**  
**Source :** [Lien source](https://openagenda.com/region-hauts-de-france/events/ledition-aurora-marque-un-tournant)

Cet événement se concentre sur l'innovation, le design et l'intelligence artificielle.
